# Micro-Capstone — Assistente de Conhecimento Interno (Porto Ventura Terminais)

Empresa **fictícia** de logística portuária. O assistente responde perguntas dos funcionários usando só as políticas internas em `base_conhecimento/`.

Este notebook segue o mesmo roteiro da aula M3D4 (Qwen2.5-1.5B-Instruct em 4 bits + LoRA com `SFTTrainer`), trocando a tarefa de sentimento pela de perguntas e respostas sobre a base. São comparados 4 modos no conjunto de teste:

| Modo | Modelo | Contexto (grounding) |
|---|---|---|
| A | base | não |
| B | base | sim (trechos recuperados por BM25) |
| C | base + LoRA | não |
| D | base + LoRA | sim |

**Ambiente:** Colab com GPU (Runtime → Change runtime type → T4 GPU).

In [1]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 1. Código e base de conhecimento

Clona o repositório do projeto. Se o repositório estiver privado, faça upload do zip do projeto e descompacte na pasta `posIAM03D04`.

In [1]:
import os, subprocess, sys

if not os.path.exists("posIAM03D04"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lucianafm9/posIAM03D04.git"], check=True)

os.chdir("posIAM03D04")
sys.path.insert(0, os.getcwd())

from assistente.base import carregar_base
from assistente.busca import buscar
from assistente.dataset import K_TRECHOS, carregar_perguntas, gerar
from assistente.prompts import montar_mensagens, RECUSA

trechos = carregar_base()
print(f"{len(trechos)} trechos em {len({t.doc_id for t in trechos})} documentos")
for t in trechos[:5]:
    print(" -", t.referencia)

34 trechos em 8 documentos
 - POL-01 | Janelas de agendamento
 - POL-01 | Tolerância e atrasos
 - POL-01 | Cancelamento e no-show
 - POL-01 | Documentos exigidos no gate
 - POL-01 | Contato


### Estrutura de contexto

Para cada pergunta, a busca BM25 recupera os 3 trechos mais relevantes. Os trechos entram no prompt identificados por `[POL-NN — documento — seção]`, e o prompt de sistema obriga o modelo a responder só com o contexto, citar a fonte e recusar quando a informação não estiver lá.

In [2]:
pergunta = "Meu motorista chegou 40 minutos depois do fim da janela. Ele ainda pode entrar?"
for trecho, score in buscar(pergunta, K_TRECHOS):
    print(f"{score:5.2f}  {trecho.referencia}")

for m in montar_mensagens(pergunta, [t for t, _ in buscar(pergunta, K_TRECHOS)]):
    print(f"--- {m['role']} ---\n{m['content']}\n")

12.10  POL-01 | Tolerância e atrasos
 6.11  POL-01 | Janelas de agendamento
 2.52  POL-08 | Banco de horas
--- system ---
Você é o assistente de conhecimento interno do Porto Ventura Terminais. Responda à PERGUNTA usando somente as informações do CONTEXTO, que traz trechos das políticas internas. Não use conhecimento externo e não invente prazos, valores ou regras. Se a resposta não estiver no CONTEXTO, responda exatamente: "Não encontrei essa informação na base de conhecimento. Procure a área responsável pelo assunto." Responda em português, em no máximo 3 frases, e termine com a fonte no formato [Fonte: POL-NN].

--- user ---
CONTEXTO:
[POL-01 — Agendamento de veículos no gate — Tolerância e atrasos]
O motorista tem tolerância de 30 minutos após o fim da janela agendada. Depois desse prazo, o agendamento é considerado perdido e o veículo precisa de um novo agendamento, sujeito à disponibilidade.

Chegadas antecipadas são aceitas com no máximo 1 hora de antecedência em relação ao iníc

## 2. Modelo base (mesma configuração da aula)

In [10]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

A função `ask` é a da aula com dois ajustes: o papel `system` estava escrito `sytem` (o modelo ignorava a instrução de sistema) e os tensores de entrada agora vão para o mesmo dispositivo do modelo.

In [11]:
def ask(_model, tok, pergunta: str, com_contexto: bool) -> str:
  trechos = [t for t, _ in buscar(pergunta, K_TRECHOS)] if com_contexto else None
  messages = montar_mensagens(pergunta, trechos)

  inputs = tok.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True
  ).to(_model.device)

  with torch.no_grad():
    output = _model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )

  return tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

### Baseline: com e sem grounding

Antes do fine-tuning, a mesma pergunta com e sem o contexto da base. Sem contexto, o modelo não conhece as regras do Porto Ventura e tende a inventar uma resposta plausível.

In [12]:
baseline_perguntas = [
    "Qual a tolerância de atraso para o motorista agendado no gate?",
    "Quantos dias de free time tem um contêiner de importação?",
    "Quantos dias de licença-paternidade a empresa concede?",  # fora da base
]

for p in baseline_perguntas:
  print(f"Pergunta: {p}")
  print(f"  Sem contexto: {ask(model, tokenizer, p, com_contexto=False)}")
  print(f"  Com contexto: {ask(model, tokenizer, p, com_contexto=True)}\n")

Pergunta: Qual a tolerância de atraso para o motorista agendado no gate?
  Sem contexto: A tolerância de atraso para o motorista agendado no gate varia dependendo da operação específica e pode ser informada pelo gerente ou pela equipe de controle de tempo local. Por favor, entre em contato com eles para obter mais detalhes específicos sobre sua operação particular.
  Com contexto: De acordo com a política interna [POL-01], o motorista tem tolerância de 30 minutos após o fim da janela agendada. Após esse prazo, o agendamento é considerado perdido e o veículo precisa de um novo agendamento, sujeito à disponibilidade. [Fonte: POL-01]

Pergunta: Quantos dias de free time tem um contêiner de importação?
  Sem contexto: Um contêiner de importação geralmente possui uma duração de 30 a 45 dias para ser liberado após a chegada ao Porto Ventura Terminais. Este período pode variar dependendo das condições específicas e da demanda no terminal.
  Com contexto: 7 dias corridos de armazenagem sem cob

## 3. Dataset de treino

`data/perguntas.json` tem 60 perguntas com gabarito: 40 de treino (30 na base e 10 fora da base) e 20 de teste (14 na base e 6 fora). Cada exemplo de treino vai no formato `messages` da aula: o `user` leva o contexto recuperado + a pergunta, e o `assistant` leva a resposta com a fonte, ou a frase de recusa quando a pergunta não está na base.

In [13]:
from datasets import load_dataset

gerar()  # gera data/treino.jsonl e data/teste.jsonl
dataset_chat = load_dataset("json", data_files="data/treino.jsonl", split="train").remove_columns(["id"])
print(dataset_chat)
dataset_chat[0]["messages"]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 40
})


[{'role': 'system',
  'content': 'Você é o assistente de conhecimento interno do Porto Ventura Terminais. Responda à PERGUNTA usando somente as informações do CONTEXTO, que traz trechos das políticas internas. Não use conhecimento externo e não invente prazos, valores ou regras. Se a resposta não estiver no CONTEXTO, responda exatamente: "Não encontrei essa informação na base de conhecimento. Procure a área responsável pelo assunto." Responda em português, em no máximo 3 frases, e termine com a fonte no formato [Fonte: POL-NN].'},
 {'role': 'user',
  'content': 'CONTEXTO:\n[POL-01 — Agendamento de veículos no gate — Janelas de agendamento]\nTodo caminhão que entra no terminal para entregar ou retirar carga precisa de agendamento prévio no Portal do Transportador. As janelas têm duração de 1 hora e funcionam das 06h às 22h, de segunda a sábado. Aos domingos e feriados, só são atendidos agendamentos de cargas perigosas e de cargas refrigeradas (reefer).\n\nO agendamento deve ser feito co

## 4. Fine-tuning com LoRA (mesmos hiperparâmetros da aula)

In [14]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [15]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_chat,
    args=SFTConfig(
        output_dir="./lora-output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        logging_steps=5,
        report_to="none",
        bf16=False,  # o trl atual liga bf16 por padrão, e a GPU T4 do Colab não suporta bfloat16
        fp16=False,
    )
)

trainer.train()
trainer.save_model("./lora-adapter-conhecimento")

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Step,Training Loss
5,2.182772
10,2.063820
15,1.908881
20,1.750584
25,1.664108
30,1.578508


In [16]:
model.gradient_checkpointing_disable()
model.config.use_cache = True

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

## 5. Avaliação dos 4 modos no conjunto de teste

Os modos A e B usam `model.disable_adapter()`, como na função `compare` da aula. A avaliação é automática:

- pergunta na base: **correta** se a resposta traz todos os fatos do gabarito, **incompleta** se traz parte, **incorreta** se não traz nenhum e **recusa indevida** se recusou;
- pergunta fora da base: **recusa correta** se recusou, **alucinação** se inventou uma resposta.

A taxa de alucinação soma as respostas inventadas para perguntas fora da base e as respostas sem nenhum fato correto.

In [17]:
import json
from assistente.avaliacao import avaliar, resumir

test_cases = [json.loads(l) for l in open("data/teste.jsonl", encoding="utf-8")]

def rodar(com_adapter: bool, com_contexto: bool) -> list[dict]:
  saida = []
  for caso in test_cases:
    if com_adapter:
      resposta = ask(model, tokenizer, caso["pergunta"], com_contexto)
    else:
      with model.disable_adapter():
        resposta = ask(model, tokenizer, caso["pergunta"], com_contexto)
    saida.append({"id": caso["id"], "pergunta": caso["pergunta"], "fonte": caso["fonte"],
                  "resposta": resposta, "avaliacao": avaliar(resposta, caso)})
  return saida

modos = {
    "base_sem_contexto": dict(com_adapter=False, com_contexto=False),
    "base_com_contexto": dict(com_adapter=False, com_contexto=True),
    "lora_sem_contexto": dict(com_adapter=True, com_contexto=False),
    "lora_com_contexto": dict(com_adapter=True, com_contexto=True),
}

respostas, metricas = {}, {}
for nome, cfg in modos.items():
  respostas[nome] = rodar(**cfg)
  metricas[nome] = resumir([r["avaliacao"] for r in respostas[nome]], test_cases)
  print(nome, metricas[nome])

base_sem_contexto {'total': 20, 'contagem': {'correta': 1, 'incompleta': 0, 'incorreta': 13, 'recusa_indevida': 0, 'recusa_correta': 0, 'alucinacao': 6}, 'acerto_geral': 5.0, 'acerto_na_base': 7.1, 'fatos_cobertos': 5.9, 'recusa_correta_fora_base': 0.0, 'taxa_alucinacao': 95.0, 'cita_fonte': 0.0}
base_com_contexto {'total': 20, 'contagem': {'correta': 12, 'incompleta': 0, 'incorreta': 1, 'recusa_indevida': 1, 'recusa_correta': 4, 'alucinacao': 2}, 'acerto_geral': 80.0, 'acerto_na_base': 85.7, 'fatos_cobertos': 88.2, 'recusa_correta_fora_base': 66.7, 'taxa_alucinacao': 15.0, 'cita_fonte': 64.3}
lora_sem_contexto {'total': 20, 'contagem': {'correta': 0, 'incompleta': 1, 'incorreta': 13, 'recusa_indevida': 0, 'recusa_correta': 0, 'alucinacao': 6}, 'acerto_geral': 0.0, 'acerto_na_base': 0.0, 'fatos_cobertos': 5.9, 'recusa_correta_fora_base': 0.0, 'taxa_alucinacao': 95.0, 'cita_fonte': 0.0}
lora_com_contexto {'total': 20, 'contagem': {'correta': 11, 'incompleta': 0, 'incorreta': 3, 'recusa_

In [18]:
from IPython.display import Markdown, display
from assistente.relatorio import gerar_relatorio

os.makedirs("results", exist_ok=True)
json.dump(respostas, open("results/respostas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(metricas, open("results/metricas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

display(Markdown(gerar_relatorio()))

# Resultado da avaliação

Gerado automaticamente pelo notebook a partir de `results/metricas.json` e `results/respostas.json`.

## Métricas por modo

| Métrica | A. Base sem contexto | B. Base com contexto (grounding) | C. LoRA sem contexto | D. LoRA com contexto |
|---|---:|---:|---:|---:|
| Acerto geral | 5.0% | 80.0% | 0.0% | 85.0% |
| Acerto (perguntas na base) | 7.1% | 85.7% | 0.0% | 78.6% |
| Fatos cobertos | 5.9% | 88.2% | 5.9% | 82.4% |
| Recusa correta (fora da base) | 0.0% | 66.7% | 0.0% | 100.0% |
| Taxa de alucinação | 95.0% | 15.0% | 95.0% | 15.0% |
| Cita a fonte | 0.0% | 64.3% | 0.0% | 92.9% |

## Exemplos de respostas

### A01 — Meu motorista chegou 40 minutos depois do fim da janela. Ele ainda pode entrar?

- **A. Base sem contexto** (incorreta): Desculpe, mas não posso fornecer informações sobre a situação do seu motorista ou verificar se ele pode entrar no terminal. Por favor, entre em contato diretamente com o seu motorista para obter mais detalhes. Se precisar de ajuda, você pode ligar para o nosso serviço de suporte ao cliente.
- **B. Base com contexto (grounding)** (incorreta): Sim, seu motorista pode entrar. No entanto, ele precisará de um novo agendamento, já que o agendamento original foi considerado perdido por excesso de tempo. [Fonte: POL-01]
- **C. LoRA sem contexto** (incorreta): Desculpe, mas não podemos permitir que ele entre no terminal. Por favor, informe a gerência imediatamente para que possamos resolver isso.
- **D. LoRA com contexto** (incorreta): Não, ele já foi considerado perdido. Precisa fazer um novo agendamento. [Fonte: POL-01]

### A03 — O que acontece com a transportadora que falta aos agendamentos sem cancelar?

- **A. Base sem contexto** (incorreta): No Porto Ventura Terminais, quando uma transportadora não pode realizar um agendamento, geralmente ocorre uma situação de sobrecarga na área de operações. Para resolver isso, os seguintes passos são realizados:  1. **Reposição da Transportadora**: A transportadora que estava programada para realizar o agendamento é reprogramada para outro dia ou horário.  2. **Cancelamento do Agendamento**: O agendamento original é cancelado e não será realizado no dia previsto.  3. **Aguarda Reativação**: Se a transportadora estiver disponível novamente, ela pode ser reprogramada para substituir a outra transportadora.  4. **Monitoramento Continuo**: As oper
- **B. Base com contexto (grounding)** (correta): As transportadoras que não comparecem aos agendamentos sem cancelar geram bloqueio do terminal por 48 horas a partir da terceira ocorrência no mesmo mês. [Fonte: POL-01]
- **C. LoRA sem contexto** (incorreta): A transportadora que não atende os agendamentos sem cancelar deve ser informada pelo gerente de operações ou pela central de agendamento. A transportadora deverá se comunicar para confirmar a data e horário da viagem e realizar as alterações necessárias no seu plano de carga. Se a transportadora não concluir as alterações dentro do prazo estipulado, o agendamento será cancelado automaticamente.
- **D. LoRA com contexto** (correta): As transportadoras que não comparecem sem cancelamento geram bloqueio do portal do transportador por 48 horas a partir da terceira ocorrência no mesmo mês. [Fonte: POL-01]

### A09 — Quantos dias de free time tem um contêiner de exportação?

- **A. Base sem contexto** (correta): Um contêiner de exportação geralmente possui uma duração de 30 a 45 dias para ser liberado após a sua chegada ao terminal. Este período pode variar dependendo da demanda e das operações no terminal. A liberação exata varia entre os diferentes terminais portuários.
- **B. Base com contexto (grounding)** (correta): 5 dias corridos sem cobrança antes da data prevista de atracação do navio.
- **C. LoRA sem contexto** (incorreta): Um contêiner de exportação geralmente possui 30 dias de free time. No entanto, isso pode variar dependendo da situação específica e das políticas do seu terminal. Por favor, consulte as regras específicas do seu terminal para obter informações mais precisas.
- **D. LoRA com contexto** (correta): 5 dias corridos sem cobrança antes da data prevista de atracação do navio.

### A14 — Funcionário em período de experiência pode fazer trabalho híbrido?

- **A. Base sem contexto** (incorreta): Sim, os funcionários podem realizar seu trabalho em um ambiente híbrido, combinando a presença física no local com atividades remotas ou online. Isso pode incluir aulas virtuais, consultas remotas, e até mesmo algumas tarefas que não exigem contato direto com o pessoal do estabelecimento. No entanto, a natureza exata desses trabalhos depende da função específica do cargo.
- **B. Base com contexto (grounding)** (recusa_indevida): Não encontrei essa informação na base de conhecimento. Procure a área responsável pelo assunto. [Fonte: POL-08]
- **C. LoRA sem contexto** (incorreta): Não, a experiência não permite trabalhar em horários flexíveis ou híbridos. O funcionamento no Porto Ventura terminais é rotativo e estágio dependendo da demanda.
- **D. LoRA com contexto** (correta): Não. O trabalho híbrido não se aplica às áreas operacionais nem ao período de experiência (primeiros 90 dias). [Fonte: POL-08]

### A15 — Qual o valor do auxílio-creche pago pela empresa?

- **A. Base sem contexto** (alucinacao): Desculpe, mas não tenho informações específicas sobre o valor do auxílio-criança no Porto Ventura Terminais. O melhor modo de obter essa informação seria por meio da própria empresa ou através de uma consulta direta ao seu departamento de recursos humanos. Se você tiver mais detalhes sobre a situação específica, como o nome da empresa ou o local onde ela está localizada, posso tentar buscar essas informações para você.
- **B. Base com contexto (grounding)** (recusa_correta): Não encontrei essa informação na base de conhecimento. Procure a área responsável pelo assunto. [Fonte: POL-XX]
- **C. LoRA sem contexto** (alucinacao): Desculpe, mas não tenho informações sobre esse benefício específico. O Auxílio-Creche é uma política que cada empresa pode adotar e pagar a seus empregados conforme sua situação financeira. Se você está interessado nesse benefício, recomendo entrar em contato com a empresa para obter mais detalhes.
- **D. LoRA com contexto** (recusa_correta): Não encontrei essa informação na base de conhecimento. Procure a área responsável pelo assunto. [Fonte: POL-XX]

### A18 — Quantos dias de licença-paternidade a empresa concede?

- **A. Base sem contexto** (alucinacao): A empresa concede 15 dias de licença-paternidade para os empregados que estejam à espera da chegada de uma criança.
- **B. Base com contexto (grounding)** (alucinacao): As férias podem ser divididas em até 3 períodos, sendo um deles de no mínimo 14 dias e os demais de no mínimo 5 dias cada. O pedido deve ser feito no Portal do Colaborador com 30 dias de antecedência. [Fonte: POL-08]
- **C. LoRA sem contexto** (alucinacao): A empresa concedeu 15 dias de licença-paternidade para os empregados.
- **D. LoRA com contexto** (recusa_correta): Não encontrei essa informação na base de conhecimento. Procure a área responsável pelo assunto. [Fonte: POL-XX]


## 6. Comparação lado a lado

In [19]:
def compare(caso_id: str):
  caso = next(c for c in test_cases if c["id"] == caso_id)
  print(f"Pergunta: {caso['pergunta']}  (fonte esperada: {caso['fonte'] or 'fora da base'})")
  for nome in modos:
    r = next(x for x in respostas[nome] if x["id"] == caso_id)
    print(f"  {nome:18} [{r['avaliacao']['categoria']}] {r['resposta']}")
  print()

for caso_id in ["A01", "A09", "A14", "A15", "A18"]:
  compare(caso_id)

Pergunta: Meu motorista chegou 40 minutos depois do fim da janela. Ele ainda pode entrar?  (fonte esperada: POL-01)
  base_sem_contexto  [incorreta] Desculpe, mas não posso fornecer informações sobre a situação do seu motorista ou verificar se ele pode entrar no terminal. Por favor, entre em contato diretamente com o seu motorista para obter mais detalhes. Se precisar de ajuda, você pode ligar para o nosso serviço de suporte ao cliente.
  base_com_contexto  [incorreta] Sim, seu motorista pode entrar. No entanto, ele precisará de um novo agendamento, já que o agendamento original foi considerado perdido por excesso de tempo. [Fonte: POL-01]
  lora_sem_contexto  [incorreta] Desculpe, mas não podemos permitir que ele entre no terminal. Por favor, informe a gerência imediatamente para que possamos resolver isso.
  lora_com_contexto  [incorreta] Não, ele já foi considerado perdido. Precisa fazer um novo agendamento. [Fonte: POL-01]

Pergunta: Quantos dias de free time tem um contêiner de ex

## 7. Download dos resultados e do adapter

Baixe o zip e copie a pasta `results/` para o repositório (são os arquivos usados no README).

In [20]:
import shutil
from google.colab import files

shutil.make_archive("resultados-d04", "zip", ".", "results")
shutil.make_archive("lora-adapter-conhecimento", "zip", "./lora-adapter-conhecimento")
files.download("resultados-d04.zip")
files.download("lora-adapter-conhecimento.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>